# RAG Experiments (Staged Evaluation)

This notebook is dedicated to reproducible experiments.

Experiment setup: 2×3 Factorial Approach
| Configuration id | Embedding Model | Chunking Strategy |
| :--- | :--- | :--- |
| C1 | Nomic-embed-text | Fixed Size |
| C2 | Nomic-embed-text | Recursive |
| C3 | Nomic-embed-text | Semantic |
| C4 | BGE-M3 | Fixed Size |
| C5 | BGE-M3 | Recursive |
| C6 | BGE-M3 | Semantic |

Following measures are being made:
- Stage A: ingestion (`ingestion_seconds`)
- Stage B: retrieval-only (`retrieval_only_seconds`)
- Stage C: answer generation from retrieved docs (`generation_only_seconds`)
- Query-to-response metric: `query_to_response_seconds = Stage B + Stage C`

It also exports retrieved chunks for manual audit. **RAGAS** (see end of notebook): the evaluator LLM uses [Ollama Cloud](https://docs.ollama.com/cloud) when `OLLAMA_API_KEY` is set (free tier has light usage limits per [pricing](https://ollama.com/pricing)); otherwise it uses the same local `ChatOllama` model as the experiment.

In [2]:
import csv
import hashlib
import json
import os
import random
import time
import uuid
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Literal

import chromadb
from IPython.display import Markdown, display
from langchain_classic.retrievers import MultiQueryRetriever
from collections import defaultdict
from langchain_community.document_loaders import DirectoryLoader, PDFPlumberLoader
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
from langchain_experimental.text_splitter import SemanticChunker

## Core Components and Setup

In [3]:
@dataclass
class ExperimentConfig:
    embedding_model: Literal["nomic-embed-text", "bge-m3"] = "nomic-embed-text"
    chunking_strategy: Literal["fixed", "recursive", "semantic"] = "semantic"
    chunk_params: Dict[str, Any] = None
    retriever_params: Dict[str, Any] = None

    llm_model: str = "llama3.2"
    llm_temperature: float = 0.1

    data_dir: str = "data_folder/"
    chroma_path: str = "chroma_database"
    collection_prefix: str = "rag_chatbot"

    logs_dir: str = "runs"
    random_seed: int = 42
    # Post-split cap for embedding APIs (Ollama nomic rejects oversized texts). None = auto.
    max_embedding_chars: int | None = None


def _defaults_if_missing(config: ExperimentConfig) -> ExperimentConfig:
    if config.chunk_params is None:
        config.chunk_params = {
            "chunk_size": 1000,
            "chunk_overlap": 200,
            "breakpoint_threshold_type": "percentile",
            "separator": "\n\n",
        }
    if config.retriever_params is None:
        config.retriever_params = {
            "k": 4,
            "fetch_k": 20,
            "lambda_mult": 0.6,
            "search_type": "mmr",
        }
    return config


def set_reproducibility(seed: int = 42) -> None:
    random.seed(seed)
    try:
        import numpy as np

        np.random.seed(seed)
    except Exception:
        pass


def _safe_collection_name(config: ExperimentConfig) -> str:
    # Separate collections by embedding/chunker to avoid embedding-dimension collisions.
    return f"{config.collection_prefix}_{config.embedding_model}_{config.chunking_strategy}".replace("-", "_")


def build_embeddings(config: ExperimentConfig):
    model = config.embedding_model.lower()

    # Passed to SentenceTransformer.encode — smaller batches use less peak RAM during ingestion.
    try:
        import torch

        _default_device = "cuda" if torch.cuda.is_available() else "cpu"
    except Exception:
        _default_device = "cpu"
    _model_kwargs = {"device": _default_device}
    _encode_kwargs = {"batch_size": 8}

    if model == "nomic-embed-text":
        try:
            from langchain_huggingface import HuggingFaceEmbeddings
        except Exception as e:
            raise RuntimeError(
                "Nomic (HF) requires HuggingFace support. Install: pip install langchain-huggingface sentence-transformers"
            ) from e
        return HuggingFaceEmbeddings(
            model_name="nomic-ai/nomic-embed-text-v1.5",
            model_kwargs=_model_kwargs,
            encode_kwargs=_encode_kwargs,
        )

    if model == "bge-m3":
        try:
            from langchain_huggingface import HuggingFaceEmbeddings
        except Exception as e:
            raise RuntimeError(
                "BGE-M3 requires HuggingFace support. Install: pip install langchain-huggingface sentence-transformers"
            ) from e

        return HuggingFaceEmbeddings(
            model_name="BAAI/bge-m3",
            model_kwargs=_model_kwargs,
            encode_kwargs=_encode_kwargs,
        )

    raise ValueError(f"Unsupported embedding model: {config.embedding_model}")


def build_chunker(config: ExperimentConfig, embeddings):
    strategy = config.chunking_strategy.lower()
    p = config.chunk_params

    if strategy == "fixed":
        return CharacterTextSplitter(
            chunk_size=int(p.get("chunk_size", 1000)),
            chunk_overlap=int(p.get("chunk_overlap", 200)),
            separator=p.get("separator", "\n\n"),
        )

    if strategy == "recursive":
        return RecursiveCharacterTextSplitter(
            chunk_size=int(p.get("chunk_size", 1000)),
            chunk_overlap=int(p.get("chunk_overlap", 200)),
        )

    if strategy == "semantic":
        return SemanticChunker(
            embeddings,
            breakpoint_threshold_type=p.get("breakpoint_threshold_type", "percentile"),
        )

    raise ValueError(f"Unsupported chunking strategy: {config.chunking_strategy}")

"""
def load_documents(data_dir: str):
    loader = DirectoryLoader(
        data_dir,
        glob="**/*.pdf",
        loader_cls=UnstructuredPDFLoader,
        loader_kwargs={"strategy": "hi_res", "mode": "single"},
        show_progress=True,
    )

    documents = loader.load()
    if not documents:
        raise RuntimeError(f"No PDF files found in '{data_dir}'.")

    for doc in documents:
        source_path = doc.metadata.get("source", "")
        if source_path and os.path.exists(source_path):
            doc.metadata["file_path"] = source_path
            with open(source_path, "rb") as f:
                doc.metadata["file_hash"] = hashlib.md5(f.read()).hexdigest()

    return documents
"""

def load_documents(data_dir: str):
    # 1. Load all pages using the fast PDFPlumberLoader
    loader = DirectoryLoader(
        data_dir,
        glob="**/*.pdf",
        loader_cls=PDFPlumberLoader,
        show_progress=True,
    )

    raw_pages = loader.load()
    if not raw_pages:
        raise RuntimeError(f"No PDF files found in '{data_dir}'.")

    # 2. Group the individual pages by their original source file
    pages_by_source = defaultdict(list)
    for page in raw_pages:
        source_path = page.metadata.get("source", "")
        pages_by_source[source_path].append(page)

    documents = []
    # 3. Stitch the pages back together for each PDF
    for source_path, pages in pages_by_source.items():
        # Ensure pages are in numerical order
        pages.sort(key=lambda x: x.metadata.get("page", 0))
        
        # Merge all page text together with newlines
        full_text = "\n".join([page.page_content for page in pages])
        
        # Build the unified metadata
        merged_metadata = {
            "source": source_path, 
            "file_path": source_path
        }
        
        if source_path and os.path.exists(source_path):
            with open(source_path, "rb") as f:
                merged_metadata["file_hash"] = hashlib.md5(f.read()).hexdigest()
                
        # 4. Create a single, unified Document for the entire PDF
        documents.append(Document(page_content=full_text, metadata=merged_metadata))

    return documents


"""def export_loaded_documents_debug(
    documents,
    output_dir: str = "runs/loaded_docs_preview",
    content_chars: int = 25000,
):
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)

    exported = []
    for i, doc in enumerate(documents, start=1):
        source_path = doc.metadata.get("source", f"doc_{i}")
        source_name = Path(source_path).stem or f"doc_{i}"
        out_path = out / f"{i:03d}_{source_name}_loaded.md"

        metadata_lines = []
        for k in sorted(doc.metadata.keys()):
            metadata_lines.append(f"- **{k}**: {doc.metadata.get(k)}")

        preview = (doc.page_content or "")[:content_chars]
        if len(doc.page_content or "") > content_chars:
            preview += "\n\n...[truncated]"

        payload = "\n".join(
            [
                f"# Loaded Document {i}",
                "",
                "## Source",
                source_path,
                "",
                "## Metadata",
                *metadata_lines,
                "",
                "## Content Preview",
                preview,
                "",
                f"_Total characters: {len(doc.page_content or '')}_",
            ]
        )
        out_path.write_text(payload, encoding="utf-8")
        exported.append(str(out_path))

    return exported
"""


def split_documents(documents, chunker):
    return chunker.split_documents(documents)



def build_vector_db(chunks, embeddings, config: ExperimentConfig):
    client = chromadb.PersistentClient(path=config.chroma_path)
    collection_name = _safe_collection_name(config)

    return Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name=collection_name,
        client=client,
    )


def build_retriever(vector_db, llm, config: ExperimentConfig):
    p = config.retriever_params
    base_retriever = vector_db.as_retriever(
        search_type=p.get("search_type", "mmr"),
        search_kwargs={
            "k": int(p.get("k", 4)),
            "fetch_k": int(p.get("fetch_k", 20)),
            "lambda_mult": float(p.get("lambda_mult", 0.6)),
        },
    )

    query_prompt = PromptTemplate(
        input_variables=["question"],
        template="""You are a query rephrasing assistant for vector search.
    Generate 1 alternative, semantically diverse versions of the user's question.
    Return only the rephrased questions, one per line.
    Original question: {question}""",
    )

    return MultiQueryRetriever.from_llm(base_retriever, llm, prompt=query_prompt)


def format_chunk_list(chunk_list):
    rows = []
    for chunk in chunk_list:
        src = os.path.basename(chunk.metadata.get("source", "Unknown"))
        rows.append(f"Document Source: {src}\nContent: {chunk.page_content}")
    return "\n\n---\n\n".join(rows)


def generate_answer_from_docs(question: str, docs, llm) -> str:
    template = """You are a helpful and accurate assistant. You answer in the same language as the question.

    Answer the question using ONLY the provided Context information below.
    If context is insufficient, explicitly say the information is not available.

    Context: {context}
    Question: {question}

    Answer:
    """
    prompt = ChatPromptTemplate.from_template(template)
    chain = prompt | llm | StrOutputParser()
    return chain.invoke({"context": format_chunk_list(docs), "question": question})


def _approx_token_count(text: str) -> int:
    return max(1, int(len(text) / 4))


def _chunk_stats(chunks):
    if not chunks:
        return {"chunk_count": 0, "avg_chunk_chars": 0.0, "avg_chunk_tokens_approx": 0.0}

    lengths = [len(c.page_content) for c in chunks]
    token_estimates = [_approx_token_count(c.page_content) for c in chunks]
    return {
        "chunk_count": len(chunks),
        "avg_chunk_chars": sum(lengths) / len(lengths),
        "avg_chunk_tokens_approx": sum(token_estimates) / len(token_estimates),
    }

# this value is stored in the run metrics (and CSV) so one can tell later which corpus a logged experiment used, without storing full paths in every row or re-reading all PDFs
def _dataset_fingerprint(documents):
    parts = []
    for d in documents:
        p = d.metadata.get("file_path", "")
        h = d.metadata.get("file_hash", "")
        parts.append(f"{p}:{h}")
    joined = "|".join(sorted(parts))
    return hashlib.md5(joined.encode("utf-8")).hexdigest()

In [4]:
# Debug procedure: inspect exactly what the loader produced.
#documents = load_documents("data_folder/")
#preview_files = export_loaded_documents_debug(documents)
#print(f"Loaded {len(documents)} document(s).")
#preview_files

## LOGGING

In [5]:
# logging single chunk for audit purposes
def export_retrieved_chunks_for_audit(
    run_id: str,
    question: str,
    docs,
    logs_dir: str,
    config_id: str = "",
    question_id: str = "",
    generated_answer: str = "",
    reference_answer: str = "",
):
    logs = Path(logs_dir)
    logs.mkdir(parents=True, exist_ok=True)

    payload = {
        "run_id": run_id,
        "config_id": config_id,
        "question_id": question_id,
        "question": question,
        "retrieved_k": len(docs),
        "chunks": [
            {
                "rank": i + 1,
                "source": d.metadata.get("source", ""),
                "file_path": d.metadata.get("file_path", ""),
                "file_hash": d.metadata.get("file_hash", ""),
                "content": d.page_content,
            }
            for i, d in enumerate(docs)
        ],
    }

    json_path = logs / f"{run_id}_retrieved_chunks.json"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)

    csv_path = logs / "retrieved_chunks_audit.csv"
    rows = []
    for c in payload["chunks"]:
        rows.append(
            {
                "run_id": run_id,
                "config_id": config_id,
                "question_id": question_id,
                "question": question,
                "rank": c["rank"],
                "source": c["source"],
                "file_path": c["file_path"],
                "file_hash": c["file_hash"],
                "content": c["content"],
                "generated_answer": generated_answer,
                "reference_answer": reference_answer,
            }
        )

    default_fields = [
        "run_id",
        "config_id",
        "question_id",
        "question",
        "rank",
        "source",
        "file_path",
        "file_hash",
        "content",
        "generated_answer",
        "reference_answer",
    ]
    write_header = not csv_path.exists()
    with open(csv_path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()) if rows else default_fields)
        if write_header:
            writer.writeheader()
        if rows:
            writer.writerows(rows)

    return str(json_path), str(csv_path)

# logging metrics from run_query_on_prepared_setup function into metrics.csv file
def log_experiment_row(config: ExperimentConfig, metrics: Dict[str, Any]):
    logs_dir = Path(config.logs_dir)
    logs_dir.mkdir(parents=True, exist_ok=True)

    run_id = metrics["run_id"]
    run_json_path = logs_dir / f"{run_id}.json"
    with open(run_json_path, "w", encoding="utf-8") as f:
        json.dump(metrics, f, ensure_ascii=False, indent=2)

    metrics_csv_path = logs_dir / "metrics.csv"
    row = {
        "run_id": run_id,
        "config_id": metrics.get("config_id", ""),
        "question_id": metrics.get("question_id", ""),
        "question": metrics.get("question", ""),
        "generated_answer": metrics.get("generated_answer", ""),
        "timestamp_utc": metrics["timestamp_utc"],
        "embedding_model": config.embedding_model,
        "chunking_strategy": config.chunking_strategy,
        "chroma_path": config.chroma_path,
        "ingestion_seconds": metrics["ingestion_seconds"],
        "retrieval_only_seconds": metrics["retrieval_only_seconds"],
        "generation_only_seconds": metrics["generation_only_seconds"],
        "query_to_response_seconds": metrics["query_to_response_seconds"],
        "ingestion_plus_query_seconds": metrics["ingestion_plus_query_seconds"],
        "chunk_count": metrics["chunk_count"],
        "avg_chunk_chars": metrics["avg_chunk_chars"],
        "avg_chunk_tokens_approx": metrics["avg_chunk_tokens_approx"],
        "k": config.retriever_params.get("k", 4),
        "fetch_k": config.retriever_params.get("fetch_k", 20),
        "dataset_fingerprint": metrics["dataset_fingerprint"],
    }

    write_header = not metrics_csv_path.exists()
    with open(metrics_csv_path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        if write_header:
            writer.writeheader()
        writer.writerow(row)

    return str(run_json_path), str(metrics_csv_path)




# SETUP (Stage A) + QUERY RUNS (Stage B & C)

In [6]:

# I have three measurements:
# ingestion (loading, chunking, embedding, und vector DB setup)
# retreieval (retrieving relevant chunks from vector DB - testing retriever performance)
# generation (answer generation from retrieved chunks - testing LLM performance)


# Stage A: ingestion is measured here - RUNS ONCE FOR CONFIG AND STAYS THE SAME FOR ALL QUESTIONS - includes performance of loading, chunking, 
# embedding, and vector DB setup. The resulting retriever and answer LLM are then reused for all questions of the experiment.
def prepare_config_setup(config: ExperimentConfig):
    """Build once per config: embeddings, chunks, vector DB, retriever, and ingestion metrics."""
    config = _defaults_if_missing(config)
    set_reproducibility(config.random_seed)

    t_a = time.perf_counter() # start time for ingestion
    embeddings = build_embeddings(config)
    chunker = build_chunker(config, embeddings)
    documents = load_documents(config.data_dir)
    chunks = split_documents(documents, chunker)
    vector_db = build_vector_db(chunks, embeddings, config)
    ingestion_seconds = time.perf_counter() - t_a

    retriever_llm = ChatOllama(model=config.llm_model, temperature=config.llm_temperature)
    retriever = build_retriever(vector_db, retriever_llm, config)
    answer_llm = ChatOllama(model=config.llm_model, temperature=config.llm_temperature)

    stats = _chunk_stats(chunks)
    setup = {
        "config": config,
        "documents": documents,
        "retriever": retriever,
        "answer_llm": answer_llm,
        "ingestion_seconds": round(ingestion_seconds, 4),
        "chunk_count": stats["chunk_count"],
        "avg_chunk_chars": round(stats["avg_chunk_chars"], 2),
        "avg_chunk_tokens_approx": round(stats["avg_chunk_tokens_approx"], 2),
        "dataset_fingerprint": _dataset_fingerprint(documents),
    }
    return setup

# Stage B and Stage C: retrieval and generation are measured here - RUNS FOR EACH QUESTION - takes the retriever and answer LLM from the prepared setup, 
# runs retrieval and generation, and logs metrics and retrieved chunks for audit. Measuring retrieval - 
def run_query_on_prepared_setup(
    setup: Dict[str, Any],
    question: str,
    config_id: str = "",
    question_id: str = "",
    log_runs: bool = True,
    reference_answer: str = "",
):
    """Run Stage B+C per question using one already-ingested config setup.

    Set log_runs=False for auxiliary passes (e.g. RAGAS) so runs/metrics.csv stays clean.
    """
    config = setup["config"]
    retriever = setup["retriever"]
    answer_llm = setup["answer_llm"]

    run_id = uuid.uuid4().hex[:12]
    ts = datetime.now(timezone.utc).isoformat()

    t_q = time.perf_counter()

    t_b = time.perf_counter()
    retrieved_docs = retriever.invoke(question)
    retrieval_only_seconds = time.perf_counter() - t_b

    t_c = time.perf_counter()
    answer = generate_answer_from_docs(question, retrieved_docs, answer_llm)
    generation_only_seconds = time.perf_counter() - t_c

    query_to_response_seconds = time.perf_counter() - t_q

    metrics = {
        "run_id": run_id,
        "config_id": config_id,
        "question_id": question_id,
        "question": question,
        "generated_answer": answer,
        "timestamp_utc": ts,
        "config": asdict(config),
        "ingestion_seconds": setup["ingestion_seconds"],
        "retrieval_only_seconds": round(retrieval_only_seconds, 4),
        "generation_only_seconds": round(generation_only_seconds, 4),
        "query_to_response_seconds": round(query_to_response_seconds, 4),
        "ingestion_plus_query_seconds": round(setup["ingestion_seconds"] + query_to_response_seconds, 4),
        "chunk_count": setup["chunk_count"],
        "avg_chunk_chars": setup["avg_chunk_chars"],
        "avg_chunk_tokens_approx": setup["avg_chunk_tokens_approx"],
        "dataset_fingerprint": setup["dataset_fingerprint"],
    }

    if log_runs:
        run_json_path, metrics_csv_path = log_experiment_row(config, metrics)
        chunks_json_path, chunks_csv_path = export_retrieved_chunks_for_audit(
            run_id=run_id,
            question=question,
            docs=retrieved_docs,
            logs_dir=config.logs_dir,
            config_id=config_id,
            question_id=question_id,
            generated_answer=answer,
            reference_answer=reference_answer,
        )
    else:
        run_json_path = metrics_csv_path = chunks_json_path = chunks_csv_path = ""

    return {
        "answer": answer,
        "retrieved_docs": retrieved_docs,
        "metrics": metrics,
        "run_json": run_json_path,
        "metrics_csv": metrics_csv_path,
        "chunks_json": chunks_json_path,
        "chunks_csv": chunks_csv_path,
    }




## Runner: Outer loop = configs (C1-C6), Inner loop = 12 queries

In [7]:

QUERIES_FILE = Path("12queries.json")


def load_queries_from_json(path: Path) -> List[Dict[str, str]]:
    payload = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(payload, list) or not payload:
        raise RuntimeError(f"No queries found in {path}")

    queries = []
    for i, row in enumerate(payload, start=1):
        if not isinstance(row, dict):
            raise RuntimeError(f"Invalid query row at index {i} in {path}")

        qid_raw = row.get("question_id")
        if not isinstance(qid_raw, int):
            raise RuntimeError(
                f"question_id must be int (row {i} in {path}); got {qid_raw!r}"
            )
        qid = str(qid_raw)
        question = str(row.get("question", "")).strip()
        if not question:
            raise RuntimeError(f"Missing question text for question_id={qid} in {path}")

        queries.append(
            {
                "question_id": qid,
                "question": question,
                "reference_answer": str(row.get("reference_answer", "")).strip(),
            }
        )

    return queries


benchmark_queries = load_queries_from_json(QUERIES_FILE)

BASE_CHROMA_DIR = Path("chroma_database")

CONFIGS = [
    {
        "config_id": "C1",
        "embedding_model": "nomic-embed-text",
        "chunking_strategy": "fixed",
        "chroma_path": str(BASE_CHROMA_DIR / "c1_nomic_fixed"),
    },
    {
        "config_id": "C2",
        "embedding_model": "nomic-embed-text",
        "chunking_strategy": "recursive",
        "chroma_path": str(BASE_CHROMA_DIR / "c2_nomic_recursive"),
    },
    {
        "config_id": "C3",
        "embedding_model": "nomic-embed-text",
        "chunking_strategy": "semantic",
        "chroma_path": str(BASE_CHROMA_DIR / "c3_nomic_semantic"),
    },
    {
        "config_id": "C4",
        "embedding_model": "bge-m3",
        "chunking_strategy": "fixed",
        "chroma_path": str(BASE_CHROMA_DIR / "c4_bge_fixed"),
    },
    {
        "config_id": "C5",
        "embedding_model": "bge-m3",
        "chunking_strategy": "recursive",
        "chroma_path": str(BASE_CHROMA_DIR / "c5_bge_recursive"),
    },
    {
        "config_id": "C6",
        "embedding_model": "bge-m3",
        "chunking_strategy": "semantic",
        "chroma_path": str(BASE_CHROMA_DIR / "c6_bge_semantic"),
    },
]

all_results = []

for cfg in CONFIGS:
    print(f"\n===== {cfg['config_id']} | {cfg['embedding_model']} + {cfg['chunking_strategy']} =====")

    config = ExperimentConfig(
        embedding_model=cfg["embedding_model"],
        chunking_strategy=cfg["chunking_strategy"],
        chunk_params={
            "chunk_size": 1000,
            "chunk_overlap": 200,
            "breakpoint_threshold_type": "percentile",
        },
        retriever_params={"k": 4, "fetch_k": 20, "lambda_mult": 0.6, "search_type": "mmr"},
        chroma_path=cfg["chroma_path"],
    )

    setup = prepare_config_setup(config)
    print(f"Ingestion done in {setup['ingestion_seconds']}s | chunks={setup['chunk_count']}")

    # Warm-up (not timed): avoid first-query latency spikes from model load/initialization.
    _ = setup["retriever"].invoke("Warm-up retrieval query.")
    _ = setup["answer_llm"].invoke("Warm-up generation. Reply with OK.")

    for q in benchmark_queries:
        result = run_query_on_prepared_setup(
            setup=setup,
            question=q["question"],
            config_id=cfg["config_id"],
            question_id=q["question_id"],
            reference_answer=q["reference_answer"],
        )
        all_results.append(result)
        print(
            f"{q['question_id']}: retrieval={result['metrics']['retrieval_only_seconds']}s, "
            f"generation={result['metrics']['generation_only_seconds']}s, "
            f"q2r={result['metrics']['query_to_response_seconds']}s"
        )

print("\nBenchmark complete.")
print("Rows logged to:", all_results[-1]["metrics_csv"] if all_results else "(none)")
print("Retrieved chunks audit:", all_results[-1]["chunks_csv"] if all_results else "(none)")


===== C1 | nomic-embed-text + fixed =====


INFO:httpx:HTTP Request: HEAD https://huggingface.co/nomic-ai/nomic-embed-text-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/nomic-ai/nomic-embed-text-v1.5/e9b6763023c676ca8431644204f50c2b100d9aab/modules.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/nomic-ai/nomic-embed-text-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/nomic-ai/nomic-embed-text-v1.5/e9b6763023c676ca8431644204f50c2b100d9aab/config_sentence_transformers.json "HTTP/1.1 200 OK"
INFO:sentence_transformers.base.model:Loading SentenceTransformer model from nomic-ai/nomic-embed-text-v1.5.
INFO:httpx:HTTP Request: HEAD https://huggingface.co/nomic-ai/nomic-embed-text-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD http

Ingestion done in 19.1459s | chunks=25


INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO:langchain_classic.retrievers.multi_query:Generated queries: ['Warm-up retrieval query.']
INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO:langchain_classic.retrievers.multi_query:Generated queries: ['- Can you provide the exact document ID and the name of the person responsible for approving the report?', '- What is the unique identifier for the document in question, along with information on its approver?']
INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


1: retrieval=1.0689s, generation=8.1293s, q2r=9.1982s


...

10: retrieval=2.5989s, generation=9.2734s, q2r=11.8723s


INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO:langchain_classic.retrievers.multi_query:Generated queries: ["The artist behind the most valuable piece in the museum's permanent exhibit.", 'Who is the creator of the most expensive artwork on display at the institution?', "What is the name of the artist responsible for the most highly valued item in the museum's permanent holdings?", 'Which artist holds the record for the most valuable work currently part of the collection?']
INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


11: retrieval=2.0918s, generation=10.164s, q2r=12.2558s


INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO:langchain_classic.retrievers.multi_query:Generated queries: ['Department with most contractors.']
INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


12: retrieval=0.5296s, generation=2.5765s, q2r=3.1061s

Benchmark complete.
Rows logged to: runs/metrics.csv
Retrieved chunks audit: runs/retrieved_chunks_audit.csv


## RAGAS

In [8]:
import os
import pandas as pd

from datasets import Dataset
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)
from ragas.llms import llm_factory
from ragas.run_config import RunConfig
from openai import OpenAI
from langchain_huggingface import HuggingFaceEmbeddings


ollama_api_key = "14bb9f8f199a4cb7933c9a2d9642e12f.5sgXLQUhTswGK8Lu8sx4gNXP"
os.environ["OLLAMA_API_KEY"] = ollama_api_key

# 1. Create an OpenAI-compatible client pointing to Ollama Cloud
ollama_client = OpenAI(
    base_url="https://ollama.com/v1",
    api_key=ollama_api_key,
)

# 2. Create RAGAS LLM via factory
ragas_llm = llm_factory(
    model="gpt-oss:20b",
    client=ollama_client,
    max_tokens=4096,
)

# 3. LangChain embeddings object (required by AnswerRelevancy in ragas==0.4.3)
ragas_embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": "mps"},  # mps stands for Apple Silicon GPU acceleration;
    encode_kwargs={"normalize_embeddings": True, "batch_size": 16},
)

# 4. Conservative run config for large cloud model
custom_run_config = RunConfig(
    timeout=60,
    max_retries=2,
    max_workers=6,
)

# 5. Prepare RAGAS dataset
# Load the transformed data
df = pd.read_csv("runs/retrieved_chunks_audit.csv")

# Group chunks by question and aggregate into a list
ragas_df = df.groupby(
    ["run_id", "config_id", "question_id", "question", "generated_answer", "reference_answer"]
)["content"].apply(list).reset_index()

# Rename columns to match what RAGAS expects
ragas_df = ragas_df.rename(
    columns={
        "generated_answer": "answer",
        "reference_answer": "ground_truth",
        "content": "contexts",
    }
)

eval_dataset = Dataset.from_pandas(ragas_df)
eval_dataset.to_csv("table_for_ragas.csv", index=False)

# 6. Use metric instances compatible with ragas==0.4.3
answer_relevancy.strictness = 1  # Avoid "returned 1 generations instead of requested 3"

result = evaluate(
    eval_dataset,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
    llm=ragas_llm,
    embeddings=ragas_embeddings,
    run_config=custom_run_config,
)

# 7. View and save results
df_results = result.to_pandas()

df_results["config_id"] = ragas_df["config_id"]
df_results["question_id"] = ragas_df["question_id"]
df_results["run_id"] = ragas_df["run_id"]

df_results.to_csv("ragas_evaluation_results.csv", index=False)
print("Evaluation complete. Saved to ragas_evaluation_results.csv")

# Quick analysis: average scores per configuration
summary_df = df_results.groupby("config_id")[["context_precision", "context_recall", "faithfulness", "answer_relevancy"]].mean()
print("\n--- Average Scores by Configuration ---")
print(summary_df)


/var/folders/s9/zfjzp2lj0p3b4chlq2yxjs180000gp/T/ipykernel_4740/3052167830.py:6: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
/var/folders/s9/zfjzp2lj0p3b4chlq2yxjs180000gp/T/ipykernel_4740/3052167830.py:6: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
/var/folders/s9/zfjzp2lj0p3b4chlq2yxjs180000gp/T/ipykernel_4740/3052167830.py:6: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import (
/var

Evaluation complete. Saved to ragas_evaluation_results.csv

--- Average Scores by Configuration ---
           context_precision  context_recall  faithfulness  answer_relevancy
config_id                                                                   
C1                  0.717460        0.791667      0.845175          0.699378
C2                  0.403472        0.597222      0.869444          0.699665
C3                  0.494444        0.625000      0.701190          0.676420
C4                  0.619004        0.833333      0.682440          0.536606
C5                  0.416204        0.791667      0.837698          0.569440
C6                  0.527546        0.583333      0.695238          0.445637
